# SCBE Pivot Training v2 — Controlled Conversation Engine

**Goal:** Generate high-quality SFT and DPO training data with measured pivot quality.

**What's different from v1:**
- Pivots have a **control metric** (relevance, growth, novelty, coherence)
- Growth parameters are tracked and expanded
- Conversations build from first principles
- ChoiceScript-style branching generates DPO preference pairs

**Corpus pillars (all IP-safe):**
1. The Six Tongues Protocol (Issac's novel)
2. Everweave game lore (12,596 paragraphs)
3. SCBE system knowledge
4. Math from first principles
5. ChoiceScript decision trees
6. Open source literature

In [ ]:
# ============================================================
# Cell 1: Install deps + mount Drive
# ============================================================
!pip install -q datasets huggingface_hub numpy

from google.colab import drive
drive.mount('/content/drive')

import json, random, hashlib, math, time
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple, Set
from pathlib import Path
from collections import defaultdict

print('Ready.')

In [ ]:
# ============================================================
# Cell 2: Pivot Quality Metric
# ============================================================

@dataclass
class PivotScore:
    """Control metric for pivot quality. Not random — measured."""
    relevance: float    # 0-1: did pivot connect to prior context?
    growth: float       # 0-1: did we learn something new?
    novelty: float      # 0-1: is this a path not yet taken?
    coherence: float    # 0-1: does conversation still make sense?

    @property
    def quality(self) -> float:
        """Weighted composite. Coherence matters most."""
        return (
            0.20 * self.relevance +
            0.25 * self.growth +
            0.20 * self.novelty +
            0.35 * self.coherence
        )

    @property
    def is_good(self) -> bool:
        return self.quality >= 0.5

    def to_dict(self):
        d = asdict(self)
        d['quality'] = self.quality
        d['is_good'] = self.is_good
        return d


# Quick test
good_pivot = PivotScore(relevance=0.8, growth=0.7, novelty=0.6, coherence=0.9)
bad_pivot = PivotScore(relevance=0.1, growth=0.1, novelty=0.9, coherence=0.1)
print(f'Good pivot: {good_pivot.quality:.3f} ({"PASS" if good_pivot.is_good else "FAIL"})')
print(f'Bad pivot:  {bad_pivot.quality:.3f} ({"PASS" if bad_pivot.is_good else "FAIL"})')

In [ ]:
# ============================================================
# Cell 3: Topic Graph with Sacred Tongue Affinity
# ============================================================

SACRED_TONGUES = {
    'KO': {'name': "Kor'aelin",  'domain': 'Authority / Control',     'neurotransmitter': 'Dopamine'},
    'AV': {'name': 'Avali',       'domain': 'Transport / Messaging',   'neurotransmitter': 'Acetylcholine'},
    'RU': {'name': 'Runethic',    'domain': 'Policy / Constraints',    'neurotransmitter': 'Serotonin'},
    'CA': {'name': 'Cassisivadan','domain': 'Compute / Encryption',    'neurotransmitter': 'Glutamate'},
    'UM': {'name': 'Umbroth',     'domain': 'Security / Secrets',      'neurotransmitter': 'GABA'},
    'DR': {'name': 'Draumric',    'domain': 'Schema / Authentication', 'neurotransmitter': 'Cortisol'},
}

@dataclass
class Topic:
    id: str
    name: str
    tongue: str  # KO/AV/RU/CA/UM/DR
    depth: int   # 0=surface, 1=intermediate, 2=deep
    keywords: List[str]
    connections: List[str]  # topic IDs this can pivot to
    questions: List[str]    # instruction prompts
    responses: List[str]    # correct response templates
    bad_responses: List[str] = field(default_factory=list)  # for DPO


# Build a topic graph from first principles
TOPICS = {
    # === MATH (first principles) ===
    'math_binary': Topic(
        id='math_binary', name='Binary Numbers', tongue='CA', depth=0,
        keywords=['binary', 'bits', 'base 2', '0 and 1'],
        connections=['math_arithmetic', 'math_logic', 'crypto_basics'],
        questions=[
            'What is binary and why do computers use it?',
            'Convert the number 42 to binary.',
            'Why is base 2 fundamental to computation?',
        ],
        responses=[
            'Binary is base-2 numbering using only 0 and 1. Computers use it because transistors have two states: on (1) and off (0). Every computation, from adding numbers to rendering graphics, reduces to operations on binary digits.',
            '42 in binary is 101010. To convert: 42 = 32+8+2 = 2^5 + 2^3 + 2^1, so bits 5,3,1 are set.',
            'Base 2 maps directly to physical switches. Any system with two distinguishable states — voltage high/low, magnetized/not, photon present/absent — can represent binary. This universality is why binary underpins all digital systems.',
        ],
        bad_responses=[
            'Binary is just 1s and 0s, nothing special.',
            '42 in binary is 110010. (Wrong answer for DPO rejection.)',
        ],
    ),
    'math_arithmetic': Topic(
        id='math_arithmetic', name='Arithmetic', tongue='CA', depth=0,
        keywords=['addition', 'multiplication', 'division', 'modular'],
        connections=['math_binary', 'math_algebra', 'math_logic'],
        questions=[
            'Explain modular arithmetic and why it matters for cryptography.',
            'What is the golden ratio and where does it appear in nature?',
        ],
        responses=[
            'Modular arithmetic is clock math — numbers wrap around at a modulus. 7 mod 3 = 1 because 7 = 2*3 + 1. It matters for cryptography because operations in a finite field are easy to compute forward but hard to reverse, which is the basis of RSA and elliptic curve cryptography.',
            'The golden ratio phi = (1+sqrt(5))/2 ≈ 1.618. It appears in spiral galaxies, sunflower seed arrangements, nautilus shells, and the Fibonacci sequence. In SCBE, phi governs Sacred Tongue weight scaling: KO=1.00, AV=phi, RU=phi^2, CA=phi^3, UM=phi^4, DR=phi^5.',
        ],
        bad_responses=[
            'Modular arithmetic is when you use a calculator.',
        ],
    ),
    'math_fft': Topic(
        id='math_fft', name='Fourier Transform', tongue='CA', depth=2,
        keywords=['FFT', 'frequency', 'spectrum', 'harmonic', 'sine wave'],
        connections=['math_arithmetic', 'audio_basics', 'scbe_spectral'],
        questions=[
            'What does the Fourier Transform do in simple terms?',
            'Why is FFT O(N log N) instead of O(N^2)?',
            'How does spectral analysis connect to the SCBE 14-layer pipeline?',
        ],
        responses=[
            'The Fourier Transform decomposes a signal into its component frequencies. Like a prism splitting white light into a rainbow, FFT splits a sound wave into the individual sine waves that make it up. Each frequency has an amplitude (how loud) and a phase (when it starts).',
            'The Cooley-Tukey FFT algorithm exploits symmetry in the DFT matrix. By splitting the N-point DFT into two N/2-point DFTs recursively, it reduces N^2 multiplications to N*log2(N). For N=1024, that is 10240 vs 1048576 — a 100x speedup.',
            'SCBE layers 9-10 use FFT for spectral coherence analysis. Layer 9 decomposes the context signal into frequency components. Layer 10 checks spin coherence — whether the phase relationships between harmonics are consistent. Inconsistent phases indicate adversarial manipulation, like a forged voice with mismatched overtones.',
        ],
        bad_responses=[
            'FFT is a fast algorithm. It makes things faster.',
        ],
    ),

    # === SCBE SYSTEM ===
    'scbe_poincare': Topic(
        id='scbe_poincare', name='Poincare Ball Model', tongue='UM', depth=1,
        keywords=['hyperbolic', 'poincare', 'trust ring', 'curvature'],
        connections=['math_fft', 'scbe_spectral', 'scbe_governance', 'scbe_tongues'],
        questions=[
            'Why does SCBE use hyperbolic geometry instead of Euclidean?',
            'What are the four trust rings in the Poincare ball?',
        ],
        responses=[
            'Hyperbolic space grows exponentially with distance from the center. In the Poincare ball, moving from radius 0.3 to 0.9 covers exponentially more "space" than 0 to 0.3. This means adversarial drift costs exponentially more the further it goes — attacks become computationally infeasible, not just expensive.',
            'CORE (r<0.3, 5ms latency): highest trust, critical ops. INNER (0.3-0.7, 30ms): standard authenticated. OUTER (0.7-0.9, 200ms): partially verified. WALL (r>=0.9): absolute deny. The exponential cost scaling of hyperbolic distance makes the WALL mathematically impenetrable.',
        ],
        bad_responses=[
            'SCBE uses hyperbolic geometry because it sounds cool.',
        ],
    ),
    'scbe_tongues': Topic(
        id='scbe_tongues', name='Sacred Tongues', tongue='KO', depth=1,
        keywords=['sacred', 'tongues', 'KO', 'AV', 'RU', 'CA', 'UM', 'DR', 'langues'],
        connections=['scbe_poincare', 'scbe_governance', 'lore_characters'],
        questions=[
            'Name the six Sacred Tongues and their neurotransmitter mappings.',
            'Why are there exactly six tongues?',
            'How do tongue weights scale and why?',
        ],
        responses=[
            "KO (Kor'aelin) = Dopamine (reward/authority). AV (Avali) = Acetylcholine (attention/transport). RU (Runethic) = Serotonin (mood/policy). CA (Cassisivadan) = Glutamate (excitation/compute). UM (Umbroth) = GABA (inhibition/security). DR (Draumric) = Cortisol (stress/authentication). Together they form a neurochemically-inspired control plane balancing excitation and inhibition.",
            'Six maps to three excitatory-inhibitory pairs: KO-UM (command-guard), AV-RU (transport-constraint), CA-DR (compute-structure). This balance mirrors biological neural regulation. Six also maps to the faces of a cube, vertices of an octahedron, and strings of a guitar.',
            'Weights scale by phi (golden ratio): KO=1.00, AV=1.62, RU=2.62, CA=4.24, UM=6.85, DR=11.09. Higher tongues are exponentially more expensive to activate, creating natural scarcity for high-privilege operations. DR (authentication/structure) costs 11x more than KO (intent/flow).',
        ],
        bad_responses=[
            'There are six tongues because six is a nice number.',
        ],
    ),

    # === LORE ===
    'lore_characters': Topic(
        id='lore_characters', name='Spiralverse Characters', tongue='DR', depth=1,
        keywords=['Marcus', 'Polly', 'Senna', 'Kael', 'Bram', 'Sera'],
        connections=['scbe_tongues', 'scbe_governance', 'lore_world'],
        questions=[
            'Who is Marcus Chen and what is his role in the Spiralverse?',
            'Describe Polly and her relationship to the Archive.',
            'What happened to Kael Thorne and why does it matter?',
        ],
        responses=[
            'Marcus Chen is a security engineer from Earth who gets pulled into Aethermoor. He becomes the first person in centuries to interact with all six Sacred Tongues simultaneously. His Earth-trained pattern recognition lets him see vulnerabilities in Aethermoor\'s governance that natives are blind to — making him both invaluable and dangerous.',
            'Polly (Polivara) is the Fifth Circle Keeper of the Archives — a raven who can shift to humanoid form. She is KO-aligned (authority/control) and serves as Marcus\'s primary guide. She is ancient, sarcastic, deeply loyal, and carries the weight of knowing how close Aethermoor\'s governance came to collapse.',
            'Kael Thorne gamed the trust model and was erased 300 years ago. But his pattern persists in the infrastructure like a ghost — 17 copies of his signature were discovered in Chapter 13, requiring a 72-drone swarm purge. He represents the fundamental question: can governance survive someone who understands it completely?',
        ],
        bad_responses=[
            'Marcus is the main character. He does stuff.',
            'Polly is a bird.',
        ],
    ),
    'lore_world': Topic(
        id='lore_world', name='Aethermoor World', tongue='DR', depth=0,
        keywords=['Aethermoor', 'Spiralverse', 'world', 'circles', 'governance'],
        connections=['lore_characters', 'scbe_tongues', 'scbe_governance'],
        questions=[
            'What is Aethermoor?',
            'How does the Circle system work?',
        ],
        responses=[
            'Aethermoor is a world where magic operates through governance protocols. There is no wild magic — every spell, transport, and transformation runs through the Sacred Tongue infrastructure. The Circles are trust tiers (First through Seventh) that determine what operations an individual can perform. Breaking Circle constraints triggers the harmonic wall.',
            'The Circle system mirrors the Poincare trust rings. First Circle is outermost (limited access). Seventh Circle is innermost (near-absolute authority). Advancement requires demonstrated mastery of progressively more Sacred Tongues. Each Circle unlocks new governance privileges but also new responsibilities and constraints.',
        ],
        bad_responses=[
            'Aethermoor is a fantasy world with magic.',
        ],
    ),

    # === AUDIO (new frontier) ===
    'audio_basics': Topic(
        id='audio_basics', name='Digital Audio Fundamentals', tongue='AV', depth=0,
        keywords=['audio', 'sample rate', 'waveform', 'Hz', 'amplitude'],
        connections=['math_fft', 'scbe_spectral', 'audio_voice'],
        questions=[
            'What is digital audio at its most fundamental level?',
            'Why is 44100 Hz a standard sample rate?',
        ],
        responses=[
            'Digital audio is a sequence of numbers representing air pressure at regular time intervals. At 44100 Hz, you record 44100 pressure measurements per second. Each measurement (sample) is a number — 16-bit means values from -32768 to 32767. Mixing two sounds is literally adding their number arrays together.',
            '44100 Hz was chosen because human hearing tops out around 20000 Hz, and the Nyquist theorem says you need at least 2x the highest frequency — so 40000 Hz minimum. 44100 was picked for compatibility with early digital recording on video tape (PAL and NTSC frame rates).',
        ],
        bad_responses=['Audio is sound files on computers.'],
    ),
}

print(f'Topic graph: {len(TOPICS)} topics')
for tid, t in TOPICS.items():
    print(f'  [{t.tongue}] {t.name} (depth {t.depth}) -> {len(t.connections)} connections')

In [ ]:
# ============================================================
# Cell 4: Controlled Pivot Engine
# ============================================================

class ControlledPivotEngine:
    """Conversation engine with measured pivots. Not random."""

    def __init__(self, topics: Dict[str, Topic]):
        self.topics = topics
        self.visit_count: Dict[str, int] = defaultdict(int)
        self.path_history: List[str] = []
        self.sft_pairs: List[Dict] = []
        self.dpo_pairs: List[Dict] = []
        self.pivot_scores: List[PivotScore] = []

    def score_pivot(self, from_id: str, to_id: str) -> PivotScore:
        """Measure pivot quality before taking it."""
        from_t = self.topics[from_id]
        to_t = self.topics[to_id]

        # Relevance: is the target actually connected?
        relevance = 1.0 if to_id in from_t.connections else 0.2

        # Shared keywords boost relevance
        shared_kw = set(from_t.keywords) & set(to_t.keywords)
        relevance = min(1.0, relevance + 0.1 * len(shared_kw))

        # Growth: does the target go deeper or explore new tongue?
        depth_delta = to_t.depth - from_t.depth
        tongue_change = 1.0 if from_t.tongue != to_t.tongue else 0.3
        growth = min(1.0, 0.3 + 0.3 * max(0, depth_delta) + 0.4 * tongue_change)

        # Novelty: have we been here before?
        visits = self.visit_count[to_id]
        novelty = max(0.0, 1.0 - 0.3 * visits)

        # Coherence: is the transition natural?
        coherence = 0.9 if to_id in from_t.connections else 0.3
        if from_t.tongue == to_t.tongue:
            coherence = min(1.0, coherence + 0.1)

        return PivotScore(
            relevance=round(relevance, 3),
            growth=round(growth, 3),
            novelty=round(novelty, 3),
            coherence=round(coherence, 3),
        )

    def best_pivot(self, current_id: str) -> Tuple[str, PivotScore]:
        """Choose the best available pivot from current topic."""
        current = self.topics[current_id]
        candidates = []

        for conn_id in current.connections:
            if conn_id in self.topics:
                score = self.score_pivot(current_id, conn_id)
                candidates.append((conn_id, score))

        if not candidates:
            for tid in self.topics:
                if tid != current_id:
                    score = self.score_pivot(current_id, tid)
                    candidates.append((tid, score))

        candidates.sort(key=lambda x: x[1].quality, reverse=True)
        return candidates[0]

    def generate_conversation(self, start_topic: str, num_turns: int = 6) -> Dict:
        """Generate a measured conversation with SFT + DPO output."""
        current_id = start_topic
        turns = []

        for turn_i in range(num_turns):
            topic = self.topics[current_id]
            self.visit_count[current_id] += 1
            self.path_history.append(current_id)

            q_idx = turn_i % len(topic.questions)
            r_idx = turn_i % len(topic.responses)
            question = topic.questions[q_idx]
            good_response = topic.responses[r_idx]

            sft = {
                'instruction': question,
                'response': good_response,
                'topic': topic.name,
                'tongue': topic.tongue,
                'depth': topic.depth,
                'turn': turn_i,
            }
            self.sft_pairs.append(sft)

            if topic.bad_responses:
                bad_idx = turn_i % len(topic.bad_responses)
                dpo = {
                    'prompt': question,
                    'chosen': good_response,
                    'rejected': topic.bad_responses[bad_idx],
                    'topic': topic.name,
                    'tongue': topic.tongue,
                }
                self.dpo_pairs.append(dpo)

            if turn_i < num_turns - 1:
                next_id, pivot_score = self.best_pivot(current_id)
                self.pivot_scores.append(pivot_score)
                turns.append({
                    'turn': turn_i,
                    'topic': topic.name,
                    'tongue': topic.tongue,
                    'question': question,
                    'pivot_to': next_id,
                    'pivot_quality': pivot_score.quality,
                    'pivot_state': 'good' if pivot_score.is_good else 'weak',
                })
                current_id = next_id
            else:
                turns.append({
                    'turn': turn_i,
                    'topic': topic.name,
                    'tongue': topic.tongue,
                    'question': question,
                    'pivot_to': None,
                    'pivot_quality': None,
                    'pivot_state': 'terminal',
                })

        avg_quality = sum(s.quality for s in self.pivot_scores) / max(1, len(self.pivot_scores))
        tongues_visited = set(self.topics[tid].tongue for tid in self.path_history)

        return {
            'turns': turns,
            'sft_count': len(self.sft_pairs),
            'dpo_count': len(self.dpo_pairs),
            'avg_pivot_quality': round(avg_quality, 4),
            'tongues_visited': sorted(tongues_visited),
            'unique_topics': len(set(self.path_history)),
        }

# Test it
engine = ControlledPivotEngine(TOPICS)
convo = engine.generate_conversation('math_binary', num_turns=8)

print(f"\nConversation: {convo['sft_count']} SFT pairs, {convo['dpo_count']} DPO pairs")
print(f"Avg pivot quality: {convo['avg_pivot_quality']:.4f}")
print(f"Tongues visited: {convo['tongues_visited']}")
print(f"Unique topics: {convo['unique_topics']}")
print()
for t in convo['turns']:
    state = t['pivot_state']
    q = t['pivot_quality']
    dest = t.get('pivot_to') or 'END'
    q_str = f'{q:.4f}' if q is not None else 'N/A'
    mark = '✓' if state == 'good' else ('○' if state == 'terminal' else '✗')
    print(f"  {mark} [{t['tongue']}] {t['topic']:<30} -> {dest:<20} (q={q_str})")

In [ ]:
# ============================================================
# Cell 5: Batch Generate + Export
# ============================================================

def batch_generate(topics, num_conversations=50, turns_per=8):
    """Generate many conversations, accumulate all SFT+DPO pairs."""
    all_sft = []
    all_dpo = []
    all_scores = []
    start_topics = list(topics.keys())

    for i in range(num_conversations):
        engine = ControlledPivotEngine(topics)
        start = start_topics[i % len(start_topics)]
        convo = engine.generate_conversation(start, num_turns=turns_per)
        all_sft.extend(engine.sft_pairs)
        all_dpo.extend(engine.dpo_pairs)
        all_scores.extend(engine.pivot_scores)

    avg_q = sum(s.quality for s in all_scores) / max(1, len(all_scores))
    good_pct = sum(1 for s in all_scores if s.is_good) / max(1, len(all_scores)) * 100

    print(f'Generated {len(all_sft)} SFT pairs, {len(all_dpo)} DPO pairs')
    print(f'Pivot quality: avg={avg_q:.4f}, good={good_pct:.1f}%')
    return all_sft, all_dpo, all_scores

sft_data, dpo_data, scores = batch_generate(TOPICS, num_conversations=100, turns_per=8)

# Save to Drive
output_dir = Path('/content/drive/MyDrive/SCBE_Training')
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / 'pivot_sft_v2.jsonl', 'w') as f:
    for row in sft_data:
        f.write(json.dumps(row) + '\n')

with open(output_dir / 'pivot_dpo_v2.jsonl', 'w') as f:
    for row in dpo_data:
        f.write(json.dumps(row) + '\n')

print(f'\nSaved to {output_dir}')
print(f'  pivot_sft_v2.jsonl ({len(sft_data)} rows)')
print(f'  pivot_dpo_v2.jsonl ({len(dpo_data)} rows)')

In [ ]:
# ============================================================
# Cell 6: Push to HuggingFace
# ============================================================

from huggingface_hub import HfApi
from google.colab import userdata

# Set your HF token in Colab secrets (key: HF_TOKEN)
try:
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = input('Enter HuggingFace token: ')

api = HfApi(token=hf_token)

DATASET_REPO = 'issdandavis/scbe-aethermoore-training-data'

# Upload SFT
api.upload_file(
    path_or_fileobj=str(output_dir / 'pivot_sft_v2.jsonl'),
    path_in_repo='pivot/pivot_sft_v2.jsonl',
    repo_id=DATASET_REPO,
    repo_type='dataset',
)

# Upload DPO
api.upload_file(
    path_or_fileobj=str(output_dir / 'pivot_dpo_v2.jsonl'),
    path_in_repo='pivot/pivot_dpo_v2.jsonl',
    repo_id=DATASET_REPO,
    repo_type='dataset',
)

print(f'Pushed to https://huggingface.co/datasets/{DATASET_REPO}')
print('  pivot/pivot_sft_v2.jsonl')
print('  pivot/pivot_dpo_v2.jsonl')

## Next Steps

1. **Expand topics** — Add math textbook content (algebra, trig, calculus, linear algebra, hyperbolic geometry)
2. **Add book chapters** — Parse The Six Tongues Protocol for character dialogue SFT pairs
3. **Add Everweave logs** — Convert game paragraphs to instruction-response format
4. **ChoiceScript converter** — Parse .txt choice files into DPO preference pairs
5. **Run training** — Use Unsloth on Colab T4 to fine-tune with combined SFT + DPO
6. **Measure** — Track pivot quality metrics across training runs